# 21 — Spatiotemporal end-to-end throughput benchmark

Benchmark the real six-variable, two-sweep, four-frame sequence loader and the full sweep-aware causal ConvGRU training step on 512 real 2013 files. Compare file batch sizes 4, 8, and 16 using 12 loader workers and bfloat16 autocast on an A100.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.8"
YEAR = 2013
FILES_PER_CLASS = 256
BATCH_SIZES = (4, 8, 16)
NUM_WORKERS = 12
BENCHMARK_PASSES = 2
SEED = 20260923
VARIABLES = (
    "DBZ",
    "KDP",
    "RHOHV",
    "VEL",
    "WIDTH",
    "ZDR",
)

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / f"tornet_detection-{PACKAGE_VERSION}-py3-none-any.whl"
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
ARCHIVE_PATH = (
    BACKUP_ROOT / f"tornet_{YEAR}.tar.gz"
)
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
    / "normalization.json"
)
RESULT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "spatiotemporal_benchmark_v1"
)
RESULT_PATH = (
    RESULT_DIRECTORY
    / "2013_end_to_end_throughput.json"
)
LOCAL_ARCHIVE_PATH = Path(
    f"/content/tornet_{YEAR}.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_spatiotemporal_benchmark"
)

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    ARCHIVE_PATH,
    NORMALIZATION_PATH,
):
    if not path.exists():
        raise FileNotFoundError(path)

if RESULT_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite {RESULT_PATH}"
    )

print("package:", PACKAGE_PATH)
print("result:", RESULT_PATH)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.8-py3-none-any.whl
result: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_benchmark_v1/2013_end_to_end_throughput.json


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.8-py3-none-any.whl'], returncode=0)

In [4]:
import json
import random
import shutil
import tarfile
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_spatiotemporal_netcdf_file,
)
from tornado_detection.models import (
    SpatiotemporalTornadoDetector,
)

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime and run all cells"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "The selected GPU does not support bfloat16"
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

if normalization.get("variables") != list(VARIABLES):
    raise AssertionError(
        "Normalization variable provenance mismatch"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
).reshape(len(VARIABLES), 2).T
stds = np.asarray(
    normalization["standard_deviations"],
    dtype=np.float32,
).reshape(len(VARIABLES), 2).T

assert means.shape == (2, 6)
assert stds.shape == (2, 6)

index = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned = assign_model_splits(
    index,
    validation_fraction=0.20,
    seed=20260913,
)
eligible = assigned.loc[
    assigned["year"].eq(YEAR)
    & assigned["model_split"].eq("train")
].copy()

files = (
    eligible.groupby(
        "archive_member",
        as_index=False,
    )
    .agg(
        positive_frames=(
            "frame_label",
            "sum",
        )
    )
)

positive_files = (
    files.loc[
        files["positive_frames"].gt(0),
        "archive_member",
    ]
    .sort_values()
    .head(FILES_PER_CLASS)
    .tolist()
)
negative_files = (
    files.loc[
        files["positive_frames"].eq(0),
        "archive_member",
    ]
    .sort_values()
    .head(FILES_PER_CLASS)
    .tolist()
)

if len(positive_files) != FILES_PER_CLASS:
    raise AssertionError(
        f"Expected {FILES_PER_CLASS} positive files; "
        f"found {len(positive_files)}"
    )

if len(negative_files) != FILES_PER_CLASS:
    raise AssertionError(
        f"Expected {FILES_PER_CLASS} negative files; "
        f"found {len(negative_files)}"
    )

member_names = (
    positive_files + negative_files
)

expected_labels = (
    eligible.loc[
        eligible["archive_member"].isin(
            member_names
        )
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
    .groupby("archive_member")[
        "frame_label"
    ]
    .apply(
        lambda values: (
            values.astype(
                np.uint8
            ).to_numpy()
        )
    )
    .to_dict()
)

print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print("torch:", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print(
    "benchmark files:",
    len(member_names),
)
print(
    "benchmark frames:",
    len(member_names) * 4,
)


AssertionError: Expected 256 positive files; found 252

In [ ]:
if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter() - copy_started
)

required_members = set(member_names)
extracted_members = set()
extraction_started = time.perf_counter()

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in required_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                f"Could not extract "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(
            member.name
        )

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing = (
    required_members
    - extracted_members
)

if missing:
    raise RuntimeError(
        f"Missing files: "
        f"{sorted(missing)[:10]}"
    )

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)
print(
    "extracted files:",
    len(extracted_members),
)

In [ ]:
class SequenceDataset(Dataset):
    def __init__(
        self,
        members,
        root,
        channel_means,
        channel_stds,
    ):
        self.members = list(members)
        self.root = root
        self.means = channel_means.reshape(
            1,
            2,
            6,
            1,
            1,
        )
        self.stds = channel_stds.reshape(
            1,
            2,
            6,
            1,
            1,
        )

    def __len__(self):
        return len(self.members)

    def __getitem__(self, index):
        member = self.members[index]
        sequence = (
            read_spatiotemporal_netcdf_file(
                self.root / member,
                variables=VARIABLES,
            )
        )

        np.testing.assert_array_equal(
            sequence.labels,
            expected_labels[member],
        )

        normalized = (
            (
                sequence.values
                - self.means
            )
            / self.stds
        ).astype(
            np.float32,
            copy=False,
        )

        return (
            torch.from_numpy(normalized),
            torch.from_numpy(
                sequence.finite_mask
            ),
            torch.from_numpy(
                sequence.range_folded_mask
            ),
            torch.from_numpy(
                sequence.coordinates
            ),
            torch.from_numpy(
                sequence.labels.astype(
                    np.float32
                )
            ),
        )


dataset = SequenceDataset(
    member_names,
    EXTRACTION_ROOT,
    means,
    stds,
)
device = torch.device("cuda")

all_labels = np.concatenate(
    [
        expected_labels[member]
        for member in member_names
    ]
).astype(np.float32)
positive_count = float(
    all_labels.sum()
)
negative_count = float(
    len(all_labels) - positive_count
)
positive_weight = (
    negative_count / positive_count
)
results = []

for batch_size in BATCH_SIZES:
    generator = (
        torch.Generator()
        .manual_seed(SEED)
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        generator=generator,
    )
    model = (
        SpatiotemporalTornadoDetector()
        .to(device)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            positive_weight,
            device=device,
        )
    )
    pass_results = []

    for pass_number in range(
        1,
        BENCHMARK_PASSES + 1,
    ):
        processed_sequences = 0
        processed_frames = 0
        loss_total = 0.0
        model.train()

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()

        for (
            values,
            finite_mask,
            range_folded_mask,
            coordinates,
            labels,
        ) in loader:
            values = values.to(
                device,
                non_blocking=True,
            )
            finite_mask = finite_mask.to(
                device,
                non_blocking=True,
            )
            range_folded_mask = (
                range_folded_mask.to(
                    device,
                    non_blocking=True,
                )
            )
            coordinates = coordinates.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                logits = model(
                    values,
                    finite_mask,
                    range_folded_mask,
                    coordinates,
                )
                loss = criterion(
                    logits,
                    labels,
                )

            loss.backward()
            optimizer.step()

            sequence_count = int(
                labels.shape[0]
            )
            frame_count = int(
                labels.numel()
            )
            processed_sequences += (
                sequence_count
            )
            processed_frames += (
                frame_count
            )
            loss_total += (
                float(loss.detach())
                * frame_count
            )

        torch.cuda.synchronize()
        elapsed = (
            time.perf_counter()
            - started
        )
        peak_allocated_gib = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )
        peak_reserved_gib = (
            torch.cuda.max_memory_reserved()
            / 1024**3
        )

        pass_result = {
            "pass": pass_number,
            "seconds": elapsed,
            "sequences": (
                processed_sequences
            ),
            "frames": processed_frames,
            "sequences_per_second": (
                processed_sequences
                / elapsed
            ),
            "frames_per_second": (
                processed_frames
                / elapsed
            ),
            "mean_loss": (
                loss_total
                / processed_frames
            ),
            "peak_allocated_gib": (
                peak_allocated_gib
            ),
            "peak_reserved_gib": (
                peak_reserved_gib
            ),
        }
        pass_results.append(
            pass_result
        )

        print(
            f"batch={batch_size} "
            f"pass={pass_number} "
            f"seconds={elapsed:.3f} "
            f"sequences/s="
            f"{processed_sequences / elapsed:.2f} "
            f"frames/s="
            f"{processed_frames / elapsed:.2f} "
            f"allocated_GiB="
            f"{peak_allocated_gib:.3f} "
            f"reserved_GiB="
            f"{peak_reserved_gib:.3f}"
        )

    measured = pass_results[-1]

    results.append(
        {
            "batch_size": batch_size,
            "worker_count": (
                NUM_WORKERS
            ),
            "passes": pass_results,
            "measured_pass": (
                BENCHMARK_PASSES
            ),
            "measured_seconds": (
                measured["seconds"]
            ),
            "measured_sequences_per_second": (
                measured[
                    "sequences_per_second"
                ]
            ),
            "measured_frames_per_second": (
                measured[
                    "frames_per_second"
                ]
            ),
            "peak_allocated_gib": (
                measured[
                    "peak_allocated_gib"
                ]
            ),
            "peak_reserved_gib": (
                measured[
                    "peak_reserved_gib"
                ]
            ),
        }
    )

    del loader
    del model
    del optimizer
    del criterion
    torch.cuda.empty_cache()


In [ ]:
import datetime

best = max(
    results,
    key=lambda row: (
        row[
            "measured_sequences_per_second"
        ]
    ),
)

canonical_train_sequences = 136_918
projected_epoch_seconds = (
    canonical_train_sequences
    / best[
        "measured_sequences_per_second"
    ]
)

artifact = {
    "artifact_kind": (
        "spatiotemporal_end_to_end_"
        "throughput_benchmark"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": (
        PACKAGE_VERSION
    ),
    "variables": list(VARIABLES),
    "sequence_shape": [
        4,
        2,
        6,
        120,
        240,
    ],
    "coordinate_count": 5,
    "year": YEAR,
    "benchmark_file_count": (
        len(member_names)
    ),
    "benchmark_frame_count": (
        len(member_names) * 4
    ),
    "positive_file_count": (
        len(positive_files)
    ),
    "negative_file_count": (
        len(negative_files)
    ),
    "worker_count": NUM_WORKERS,
    "bfloat16_autocast": True,
    "results": results,
    "best_batch_size": (
        best["batch_size"]
    ),
    "best_sequences_per_second": (
        best[
            "measured_sequences_per_second"
        ]
    ),
    "best_frames_per_second": (
        best[
            "measured_frames_per_second"
        ]
    ),
    "canonical_train_sequence_count": (
        canonical_train_sequences
    ),
    "projected_training_epoch_seconds": (
        projected_epoch_seconds
    ),
    "projected_training_epoch_minutes": (
        projected_epoch_seconds
        / 60.0
    ),
    "copy_seconds": copy_seconds,
    "extraction_seconds": (
        extraction_seconds
    ),
}

RESULT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)
RESULT_PATH.write_text(
    json.dumps(
        artifact,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        artifact,
        indent=2,
        sort_keys=True,
    )
)
print("wrote:", RESULT_PATH)


In [ ]:
shutil.rmtree(EXTRACTION_ROOT)

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()
assert RESULT_PATH.is_file()

print(
    "Removed all Colab-local "
    "benchmark artifacts"
)
print("Preserved:", RESULT_PATH)
